# Investigating pretrained model

The [pretrained mopadi models are available on huggingface](https://huggingface.co/KatherLab/MoPaDi). But in the repo, the autoencoder and the diffusion model are saved as a combined model checkpoint. The question is now, if it is possible to separate them from another to just use the diffusion model and not the autoencoder.

In [7]:
import os
from huggingface_hub import hf_hub_download
import torch
from collections import Counter, defaultdict

In [4]:
token = os.getenv("HF_TOKEN")

In [6]:
# download model files
autoenc_model_path = hf_hub_download(
    repo_id="KatherLab/MoPaDi",
    filename="brca_512_model/autoenc.ckpt",
    cache_dir="/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/"
)
print(f"Autoencoder's checkpoint downloaded to: {autoenc_model_path}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Autoencoder's checkpoint downloaded to: /mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/models--KatherLab--MoPaDi/snapshots/5d8e775e24473c5d8f4c0c57fd5c865c3c2a4aab/brca_512_model/autoenc.ckpt


In [18]:
orig = "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/models--KatherLab--MoPaDi/snapshots/5d8e775e24473c5d8f4c0c57fd5c865c3c2a4aab/brca_512_model/autoenc.ckpt"
ckpt = torch.load(orig, map_location="cpu")
sd = ckpt.get("state_dict", ckpt)   # adapt for PL-style checkpoints
keys = list(sd.keys())
print("total keys:", len(keys))

total keys: 1413


In [19]:
keys = list(sd.keys())

# ✅ Autoencoder: Only first_stage_model and its submodules
autoenc_prefixes = [
    "first_stage_model",
    "encoder",
    "decoder",
    "quant_conv",
    "post_quant_conv",
    "model.first_stage_model",
    "model.encoder",
    "model.decoder",
    "model.quant_conv",
    "model.post_quant_conv",
]

# ✅ Diffusion model: Only keys under 'model' that are NOT autoencoder
diffusion_prefixes = [
    "model",
    "model.unet",
    "model.diffusion_model",
    "model.ema_model",
    "model.ema_model.model",
]

# Helper: Check if key starts with any prefix
def starts_with_any(k, prefixes):
    return any(k.startswith(p) for p in prefixes)

def is_autoencoder(k):
    return starts_with_any(k, autoenc_prefixes)

def is_diffusion(k):
    # Only include if it starts with diffusion prefix AND is NOT autoencoder
    if not starts_with_any(k, diffusion_prefixes):
        return False
    if is_autoencoder(k):
        return False  # Exclude autoencoder keys
    return True

# Split main model
auto_keys = [k for k in keys if is_autoencoder(k)]
diff_keys = [k for k in keys if is_diffusion(k)]

# Handle EMA model
ema_keys = [k for k in keys if k.startswith("ema_model.")]
ema_auto_keys = [k for k in ema_keys if is_autoencoder(k)]
ema_diff_keys = [k for k in ema_keys if is_diffusion(k)]

# ✅ Add EMA keys to the correct split
auto_keys += ema_auto_keys
diff_keys += ema_diff_keys

# Deduplicate
auto_keys = list(set(auto_keys))
diff_keys = list(set(diff_keys))

# Check
overlap = set(auto_keys) & set(diff_keys)
leftover = set(keys) - (set(auto_keys) | set(diff_keys))

print(f"Autoencoder keys: {len(auto_keys)}")
print(f"Diffusion keys: {len(diff_keys)}")
print(f"Overlap: {len(overlap)}")
print(f"Leftover: {len(leftover)}")

Autoencoder keys: 116
Diffusion keys: 590
Overlap: 0
Leftover: 707


In [17]:
# Build state dicts
auto_sd = {k: sd[k] for k in auto_keys}
diff_sd = {k: sd[k] for k in diff_keys}

# Save
torch.save({"state_dict": auto_sd}, "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/autoencoder_split.ckpt")
torch.save({"state_dict": diff_sd}, "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/diffusion_split.ckpt")